# PulseChain: Improved Delay-Risk Model
This notebook replaces the first model with a more rigorous one:
1. **Stricter leakage control.** History features only use shipments already *delivered* at least 60 days before the current shipment's scheduled date.
2. **Better features.** Recent (rolling-window) vendor and country late rates, vendor-country history, and system-wide recent late rate.
3. **Time-series cross-validation** on the development set instead of one split.
4. **Honest baselines.** A simple rule (vendor's recent late rate) is compared with Logistic Regression and LightGBM.
5. **Thresholds chosen on validation predictions**, not on the test set.
6. **Calibration**, so a score can be read as an approximate probability.

Test set: the newest 20% of shipments, never used for tuning.

## 0. Setup (Colab)

In [ ]:
!pip install -q lightgbm shap
import os
os.makedirs("export/models", exist_ok=True)
os.makedirs("export/data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
from google.colab import files
uploaded = files.upload()
DATA = list(uploaded.keys())[0]
print("Using:", DATA)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
import sklearn
import lightgbm as lgb
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, fbeta_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42

## 1. Load data and build the target

In [ ]:
COUNTRIES = ["Rwanda", "Uganda", "Tanzania", "Kenya", "Burundi", "South Sudan", "Ethiopia"]
df = pd.read_csv(DATA, encoding="latin-1")
df = df[df["Country"].isin(COUNTRIES)].copy()
for c in ["PO Sent to Vendor Date", "Scheduled Delivery Date", "Delivered to Client Date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["delay_days"] = (df["Delivered to Client Date"] - df["Scheduled Delivery Date"]).dt.days
df["target_late"] = (df["delay_days"] > 0).astype(int)
df["weight_kg"] = pd.to_numeric(df["Weight (Kilograms)"], errors="coerce")
df["freight_cost_usd"] = pd.to_numeric(df["Freight Cost (USD)"], errors="coerce")
df["planned_lead_time_days"] = (df["Scheduled Delivery Date"] - df["PO Sent to Vendor Date"]).dt.days
df = df.sort_values("Scheduled Delivery Date").reset_index(drop=True)
print(df.shape, "late share: %.3f" % df["target_late"].mean())

## 2. Leakage-safe history features
For each shipment, the history only includes earlier shipments whose **delivered date is at least `lag_days` before this shipment's scheduled date**. This mimics what could be known when the shipment is planned, because a shipment's outcome is only known once it is delivered.

In [ ]:
def add_history(frame, lag_days=60, vendor_window=20, country_window=50, global_window=100):
    d = frame.copy()
    sched = d["Scheduled Delivery Date"].values.astype("datetime64[D]")
    deliv = d["Delivered to Client Date"].values.astype("datetime64[D]")
    y = d["target_late"].values.astype(float)
    cutoff = sched - np.timedelta64(lag_days, "D")

    def groups(keys):
        out = {}
        for k in np.unique(keys):
            idx = np.where(keys == k)[0]
            idx = idx[np.argsort(deliv[idx], kind="stable")]
            out[k] = (deliv[idx], y[idx])
        return out

    vendor = d["Vendor"].values
    country = d["Country"].values
    pair = (d["Vendor"] + "|" + d["Country"]).values
    gv, gc, gp = groups(vendor), groups(country), groups(pair)
    order = np.argsort(deliv, kind="stable")
    deliv_all, y_all = deliv[order], y[order]

    names = ["vendor_past_late_rate", "vendor_recent_late_rate", "country_past_late_rate",
             "country_recent_late_rate", "vendor_country_late_rate", "global_recent_late_rate"]
    cols = {k: np.full(len(d), np.nan) for k in names}
    cols["vendor_shipment_count_so_far"] = np.zeros(len(d))
    cols["vendor_country_count"] = np.zeros(len(d))
    base = y.mean()
    for i in range(len(d)):
        k = np.searchsorted(deliv_all, cutoff[i], side="right")
        g_rate = y_all[:k].mean() if k > 0 else base
        cols["global_recent_late_rate"][i] = y_all[max(0, k - global_window):k].mean() if k > 0 else base
        dv, yv = gv[vendor[i]]
        kv = np.searchsorted(dv, cutoff[i], side="right")
        v_rate = yv[:kv].mean() if kv > 0 else g_rate
        cols["vendor_past_late_rate"][i] = v_rate
        cols["vendor_recent_late_rate"][i] = yv[max(0, kv - vendor_window):kv].mean() if kv > 0 else g_rate
        cols["vendor_shipment_count_so_far"][i] = kv
        dc, yc = gc[country[i]]
        kc = np.searchsorted(dc, cutoff[i], side="right")
        c_rate = yc[:kc].mean() if kc > 0 else g_rate
        cols["country_past_late_rate"][i] = c_rate
        cols["country_recent_late_rate"][i] = yc[max(0, kc - country_window):kc].mean() if kc > 0 else g_rate
        dp, yp = gp[pair[i]]
        kp = np.searchsorted(dp, cutoff[i], side="right")
        cols["vendor_country_late_rate"][i] = yp[:kp].mean() if kp > 0 else (v_rate if kv > 0 else c_rate)
        cols["vendor_country_count"][i] = kp
    for k, v in cols.items():
        d[k] = v
    return d


def add_engineered(frame):
    d = frame.copy()
    d["has_lead_time"] = d["planned_lead_time_days"].notna().astype(int)
    d["value_per_unit"] = d["Line Item Value"] / d["Line Item Quantity"].replace(0, np.nan)
    d["freight_per_kg"] = d["freight_cost_usd"] / d["weight_kg"].replace(0, np.nan)
    d["scheduled_month"] = d["Scheduled Delivery Date"].dt.month
    return d


LAG_DAYS = 60
data = add_engineered(add_history(df, LAG_DAYS))
split = int(len(data) * 0.8)
dev = data.iloc[:split].reset_index(drop=True)
test = data.iloc[split:].reset_index(drop=True)
y_dev = dev["target_late"].values
y_test = test["target_late"].values
print("Development:", dev.shape, "late %.3f" % y_dev.mean(), "| until", dev["Scheduled Delivery Date"].max().date())
print("Test:       ", test.shape, "late %.3f" % y_test.mean(), "| from", test["Scheduled Delivery Date"].min().date())

### Sensitivity: does the history lag change how informative the features are?
AUC of the single feature `vendor_recent_late_rate` on the test set, for different lags.

In [ ]:
rows = []
for lag in [0, 30, 60, 90, 120]:
    t = add_history(df, lag).iloc[split:]
    rows.append({"lag_days": lag,
                 "vendor_recent AUC": roc_auc_score(t["target_late"], t["vendor_recent_late_rate"]),
                 "vendor_past AUC": roc_auc_score(t["target_late"], t["vendor_past_late_rate"]),
                 "country_past AUC": roc_auc_score(t["target_late"], t["country_past_late_rate"])})
pd.DataFrame(rows).round(3)

## 3. Feature sets and ablation
Which features actually help? Each set is scored with time-series cross-validation on the **development set only** (5 expanding-window folds: train on the past, validate on the next period). A Logistic Regression is used as a fast, stable probe.

**Selection rule, fixed before looking at any test results:** keep the sets whose mean CV PR-AUC is within one standard error of the best, then pick the one with the highest PR-AUC on the *latest* fold, because the most recent period best resembles the future. Count features (how many earlier shipments a vendor has) and the month grow with time, so a model can use them as a clock; the compact sets G, H and I avoid them. In the latest fold, the late rate was far higher than in earlier folds, a sign of drift.

In [ ]:
ALL_CAT = ["Country", "Shipment Mode", "Vendor INCO Term", "Fulfill Via", "Product Group"]
BASE_NUM = ["planned_lead_time_days", "weight_kg", "freight_cost_usd", "Line Item Quantity", "Line Item Value"]
ORIGINAL_HISTORY = ["vendor_past_late_rate", "country_past_late_rate", "vendor_shipment_count_so_far"]
RECENCY = ["vendor_recent_late_rate", "country_recent_late_rate", "global_recent_late_rate",
           "vendor_country_late_rate", "vendor_country_count"]
ENGINEERED = ["has_lead_time", "value_per_unit", "freight_per_kg", "scheduled_month"]

RECENT_ONLY = ["vendor_recent_late_rate", "country_recent_late_rate", "global_recent_late_rate"]
FEATURE_SETS = {
    "A original history": (BASE_NUM + ORIGINAL_HISTORY, ALL_CAT),
    "B + recency features": (BASE_NUM + ORIGINAL_HISTORY + RECENCY, ALL_CAT),
    "C + engineered": (BASE_NUM + ORIGINAL_HISTORY + RECENCY + ENGINEERED, ALL_CAT),
    "E original history, no counts": (BASE_NUM + ["vendor_past_late_rate", "country_past_late_rate"], ALL_CAT),
    "F recency, no counts": (BASE_NUM + ["vendor_past_late_rate", "country_past_late_rate"] + RECENT_ONLY + ["vendor_country_late_rate"], ALL_CAT),
    "G recent rates only": (RECENT_ONLY, []),
    "H recent rates + vendor history": (RECENT_ONLY + ["vendor_past_late_rate"], []),
    "I vendor recent rate only": (["vendor_recent_late_rate"], []),
}

tscv = TimeSeriesSplit(n_splits=5)


def make_pre(num, cat, scale):
    num_steps = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
    parts = [("num", Pipeline(num_steps), num)]
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat))
    return ColumnTransformer(parts)


def make_lgbm(num, cat, **params):
    settings = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8,
                    subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=RANDOM_STATE,
                    n_jobs=1, verbose=-1)
    settings.update(params)
    return Pipeline([("pre", make_pre(num, cat, False)), ("clf", lgb.LGBMClassifier(**settings))])


def make_logreg(num, cat):
    return Pipeline([("pre", make_pre(num, cat, True)),
                     ("clf", LogisticRegression(max_iter=3000, C=1.0, random_state=RANDOM_STATE))])


def cv_run(factory, num, cat):
    cols = num + cat
    oof = np.full(len(dev), np.nan)
    aucs, aps = [], []
    for tr, va in tscv.split(dev):
        model = factory(num, cat)
        model.fit(dev.iloc[tr][cols], y_dev[tr])
        p = model.predict_proba(dev.iloc[va][cols])[:, 1]
        oof[va] = p
        aucs.append(roc_auc_score(y_dev[va], p))
        aps.append(average_precision_score(y_dev[va], p))
    return {"auc_mean": np.mean(aucs), "auc_std": np.std(aucs), "ap_mean": np.mean(aps), "ap_std": np.std(aps),
            "last_auc": aucs[-1], "last_ap": aps[-1], "oof": oof}


ablation = []
for name, (num, cat) in FEATURE_SETS.items():
    r = cv_run(make_logreg, num, cat)
    ablation.append({"Feature set": name, "CV ROC-AUC": r["auc_mean"], "AUC std": r["auc_std"],
                     "CV PR-AUC": r["ap_mean"], "PR-AUC std": r["ap_std"],
                     "Latest fold ROC-AUC": r["last_auc"], "Latest fold PR-AUC": r["last_ap"]})
ablation = pd.DataFrame(ablation).set_index("Feature set").round(3)
ablation


In [ ]:
best_mean = ablation["CV PR-AUC"].max()
se = ablation["PR-AUC std"] / np.sqrt(tscv.get_n_splits())
best_se = se[ablation["CV PR-AUC"].idxmax()]
candidates = ablation[ablation["CV PR-AUC"] >= best_mean - best_se]
best_set = candidates["Latest fold PR-AUC"].idxmax()
FEATURES_NUM, FEATURES_CAT = FEATURE_SETS[best_set]
FEATURES = FEATURES_NUM + FEATURES_CAT
print("Candidates within one standard error of the best:", list(candidates.index))
print("Selected feature set:", best_set)
print(len(FEATURES), "features:", FEATURES)

## 4. Models compared
- **Rule baseline:** score = vendor's recent late rate (no machine learning).
- **Logistic Regression** (linear baseline).
- **LightGBM** with default settings.
- **LightGBM tuned** by random search with time-series cross-validation, optimising PR-AUC.

In [ ]:
rule_col = "vendor_recent_late_rate"
lr_cv = cv_run(make_logreg, FEATURES_NUM, FEATURES_CAT)
lgbm_cv = cv_run(make_lgbm, FEATURES_NUM, FEATURES_CAT)
print("Logistic Regression CV AUC %.3f  PR-AUC %.3f" % (lr_cv["auc_mean"], lr_cv["ap_mean"]))
print("LightGBM (default)  CV AUC %.3f  PR-AUC %.3f" % (lgbm_cv["auc_mean"], lgbm_cv["ap_mean"]))

In [ ]:
search = RandomizedSearchCV(
    make_lgbm(FEATURES_NUM, FEATURES_CAT),
    param_distributions={
        "clf__num_leaves": [4, 7, 15, 31],
        "clf__n_estimators": [100, 200, 400, 600],
        "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "clf__min_child_samples": [10, 20, 40, 80],
        "clf__subsample": [0.6, 0.8, 1.0],
        "clf__colsample_bytree": [0.5, 0.7, 0.9],
        "clf__reg_lambda": [0, 1, 5, 20],
        "clf__max_depth": [-1, 3, 5],
    },
    n_iter=30, scoring="average_precision", cv=tscv, random_state=RANDOM_STATE, n_jobs=-1, refit=True)
search.fit(dev[FEATURES], y_dev)
print("Best CV PR-AUC: %.3f" % search.best_score_)
print(search.best_params_)

In [ ]:
tuned_oof = np.full(len(dev), np.nan)
tuned_aucs, tuned_aps = [], []
for tr, va in tscv.split(dev):
    m = clone(search.best_estimator_)
    m.fit(dev.iloc[tr][FEATURES], y_dev[tr])
    p = m.predict_proba(dev.iloc[va][FEATURES])[:, 1]
    tuned_oof[va] = p
    tuned_aucs.append(roc_auc_score(y_dev[va], p))
    tuned_aps.append(average_precision_score(y_dev[va], p))
tuned_cv = {"auc_mean": np.mean(tuned_aucs), "auc_std": np.std(tuned_aucs),
            "ap_mean": np.mean(tuned_aps), "ap_std": np.std(tuned_aps), "oof": tuned_oof}
print("LightGBM (tuned) CV AUC %.3f  PR-AUC %.3f" % (tuned_cv["auc_mean"], tuned_cv["ap_mean"]))

## 5. Choose thresholds on validation predictions (not the test set)
The out-of-fold predictions from cross-validation are used to find the score at which **80% of truly late shipments are caught** (the recall target from the proposal).

In [ ]:
RECALL_TARGET = 0.80
mask = ~np.isnan(tuned_oof)
y_val = y_dev[mask]


def threshold_for_recall(y, p, target):
    late_scores = np.sort(p[y == 1])
    idx = int(np.floor((1 - target) * len(late_scores)))
    return float(late_scores[idx])


models = {
    "Rule: vendor recent late rate": {"val": dev[rule_col].values[mask], "test": test[rule_col].values, "cv": None},
    "Logistic Regression": {"val": lr_cv["oof"][mask], "cv": lr_cv},
    "LightGBM (default)": {"val": lgbm_cv["oof"][mask], "cv": lgbm_cv},
    "LightGBM (tuned)": {"val": tuned_oof[mask], "cv": tuned_cv},
}
lr_final = make_logreg(FEATURES_NUM, FEATURES_CAT).fit(dev[FEATURES], y_dev)
lgbm_final = make_lgbm(FEATURES_NUM, FEATURES_CAT).fit(dev[FEATURES], y_dev)
tuned_final = search.best_estimator_
models["Logistic Regression"]["test"] = lr_final.predict_proba(test[FEATURES])[:, 1]
models["LightGBM (default)"]["test"] = lgbm_final.predict_proba(test[FEATURES])[:, 1]
models["LightGBM (tuned)"]["test"] = tuned_final.predict_proba(test[FEATURES])[:, 1]
for name, m in models.items():
    m["threshold"] = threshold_for_recall(y_val, m["val"], RECALL_TARGET)
    print("%-32s threshold %.3f" % (name, m["threshold"]))

fitted = {"Logistic Regression": lr_final, "LightGBM (default)": lgbm_final, "LightGBM (tuned)": tuned_final}
final_name = max(fitted, key=lambda n: models[n]["cv"]["ap_mean"])
final_model = fitted[final_name]
final_oof = models[final_name]["cv"]["oof"]
print("Final model (highest cross-validated PR-AUC, test set not used):", final_name)

## 6. Results on the held-out test set

In [ ]:
def operating_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    return {"Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred),
            "F1": f1_score(y, pred), "F2 (recall-weighted)": fbeta_score(y, pred, beta=2),
            "Flagged share": pred.mean()}


def top_fraction(y, p, frac):
    k = max(1, int(round(len(y) * frac)))
    idx = np.argsort(-p, kind="stable")[:k]
    precision = y[idx].mean()
    return precision, y[idx].sum() / y.sum(), precision / y.mean()


def bootstrap_auc(y, p, n=1000):
    rng = np.random.default_rng(RANDOM_STATE)
    values = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        values.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(values, [2.5, 97.5])


rows = []
for name, m in models.items():
    p = m["test"]
    ci = bootstrap_auc(y_test, p)
    ops = operating_metrics(y_test, p, m["threshold"])
    prec20, rec20, lift20 = top_fraction(y_test, p, 0.20)
    rows.append({"Model": name, "ROC-AUC": roc_auc_score(y_test, p), "AUC 95% CI": "%.2f-%.2f" % (ci[0], ci[1]),
                 "PR-AUC": average_precision_score(y_test, p),
                 "CV ROC-AUC": m["cv"]["auc_mean"] if m["cv"] else np.nan,
                 **ops, "Top-20% precision": prec20, "Top-20% recall": rec20, "Lift@20%": lift20})
results = pd.DataFrame(rows).set_index("Model")
print("Late share in test set: %.3f (a random ranking gives PR-AUC of about this value)" % y_test.mean())
results.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m["test"])
    ax[0].plot(fpr, tpr, label="%s (%.2f)" % (name, roc_auc_score(y_test, m["test"])))
    prec, rec, _ = precision_recall_curve(y_test, m["test"])
    ax[1].plot(rec, prec, label=name)
    order = np.argsort(-m["test"], kind="stable")
    gain = np.cumsum(y_test[order]) / y_test.sum()
    ax[2].plot(np.arange(1, len(y_test) + 1) / len(y_test), gain, label=name)
ax[0].plot([0, 1], [0, 1], "k--", lw=1)
ax[0].set_title("ROC curve"); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("Recall"); ax[0].legend(fontsize=8)
ax[1].axhline(y_test.mean(), color="k", ls="--", lw=1)
ax[1].set_title("Precision-recall curve"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[2].plot([0, 1], [0, 1], "k--", lw=1)
ax[2].set_title("Gain: share of late shipments caught"); ax[2].set_xlabel("Share of shipments reviewed (highest risk first)"); ax[2].set_ylabel("Share of late shipments caught")
plt.tight_layout(); plt.savefig("figures/06_model_comparison.png", dpi=150); plt.show()

### Fair comparison at equal review capacity
If a coordinator can only review a fixed share of shipments (highest risk first), how many late shipments does each model catch? This is fairer than recall at a threshold, because recall alone ignores how many shipments are flagged.

In [ ]:
table = []
for frac in [0.10, 0.20, 0.30, 0.40, 0.50]:
    row = {"Shipments reviewed": "%d%%" % (frac * 100)}
    for name, m in models.items():
        prec, rec, lift = top_fraction(y_test, m["test"], frac)
        row[name] = "%.0f%% caught (%.1fx)" % (rec * 100, lift)
    table.append(row)
pd.DataFrame(table).set_index("Shipments reviewed")

## 7. Calibration
Turning the raw score into an approximate probability using the out-of-fold predictions (isotonic regression).

In [ ]:
p_val = final_oof[mask]
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p_val, y_val)
p_test_raw = models[final_name]["test"]
p_test_cal = iso.predict(p_test_raw)
print("Brier score (lower is better), raw: %.4f  calibrated: %.4f" % (brier_score_loss(y_test, p_test_raw), brier_score_loss(y_test, p_test_cal)))
print("Mean predicted (calibrated): %.3f  actual late share: %.3f" % (p_test_cal.mean(), y_test.mean()))

bins = pd.qcut(p_test_raw, 5, duplicates="drop")
rel = pd.DataFrame({"raw": p_test_raw, "cal": p_test_cal, "actual": y_test}).groupby(bins).mean()
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(rel["cal"], rel["actual"], "o-", label="Calibrated score")
ax.plot(rel["raw"], rel["actual"], "s-", label="Raw score")
ax.plot([0, 0.6], [0, 0.6], "k--", lw=1)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual late rate"); ax.set_title("Reliability (test set, 5 bins)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/07_calibration.png", dpi=150); plt.show()

## 8. Explainability (SHAP) for the final model

In [ ]:
import shap
pre = final_model.named_steps["pre"]
clf = final_model.named_steps["clf"]
Xt = pre.transform(test[FEATURES])
names = [n.replace("num__", "").replace("cat__", "") for n in pre.get_feature_names_out()]
if hasattr(clf, "coef_"):
    coef = pd.Series(clf.coef_[0], index=names).sort_values()
    coef.plot.barh(figsize=(7, max(3, 0.4 * len(coef))), color="#059669")
    plt.title("Standardised coefficients (log-odds per 1 std of the feature)")
else:
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(np.asarray(Xt))
    sv = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv, Xt, feature_names=names, show=False)
plt.tight_layout(); plt.savefig("figures/08_shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()

## 9. Export everything the app needs
Creates `pulsechain_model_export.zip` containing the model, metrics, thresholds, calibration, feature configuration, the test shipments with features, and a history snapshot for new predictions.

In [ ]:
def history_snapshot(frame, vendor_window=20, country_window=50, global_window=100):
    f = frame.sort_values("Delivered to Client Date")
    snap = {"global_late_rate": float(f["target_late"].mean()),
            "global_recent_late_rate": float(f["target_late"].tail(global_window).mean()),
            "vendors": {}, "countries": {}, "pairs": {}}
    for v, g in f.groupby("Vendor"):
        snap["vendors"][v] = {"past_late_rate": float(g["target_late"].mean()),
                              "recent_late_rate": float(g["target_late"].tail(vendor_window).mean()), "count": int(len(g))}
    for c, g in f.groupby("Country"):
        snap["countries"][c] = {"past_late_rate": float(g["target_late"].mean()),
                                "recent_late_rate": float(g["target_late"].tail(country_window).mean()), "count": int(len(g))}
    for (v, c), g in f.groupby(["Vendor", "Country"]):
        snap["pairs"][v + "|" + c] = {"late_rate": float(g["target_late"].mean()), "count": int(len(g))}
    return snap


import joblib
joblib.dump(final_model, "export/models/model_pipeline.joblib")
joblib.dump(tuned_final, "export/models/lgbm_pipeline.joblib")
joblib.dump(lr_final, "export/models/logreg_pipeline.joblib")

metrics = results.reset_index().rename(columns={"Model": "Model"})
export_metrics = pd.DataFrame({
    "Model": metrics["Model"], "Precision": metrics["Precision"], "Recall": metrics["Recall"], "F1": metrics["F1"],
    "F2 (recall-weighted)": metrics["F2 (recall-weighted)"], "ROC-AUC": metrics["ROC-AUC"], "PR-AUC": metrics["PR-AUC"],
    "Flagged share": metrics["Flagged share"], "Top-20% precision": metrics["Top-20% precision"],
    "Top-20% recall": metrics["Top-20% recall"], "Lift@20%": metrics["Lift@20%"]}).round(4)
export_metrics.to_csv("export/models/initial_metrics.csv", index=False)

thresholds = {"mode": "percentile", "medium": round(float(np.percentile(p_val, 60)), 4),
              "high": round(float(np.percentile(p_val, 85)), 4), "medium_share": 0.25, "high_share": 0.15,
              "source": "percentiles of out-of-fold validation scores on the development set (test set not used)"}
json.dump(thresholds, open("export/models/thresholds.json", "w"), indent=1)

grid = np.linspace(0, float(max(p_val.max(), p_test_raw.max())) + 0.05, 101)
json.dump({"x": [round(float(v), 5) for v in grid], "y": [round(float(v), 5) for v in iso.predict(grid)]},
          open("export/models/calibration.json", "w"))

feature_config = {"numeric": FEATURES_NUM, "categorical": FEATURES_CAT, "lag_days": LAG_DAYS,
                  "windows": {"vendor": 20, "country": 50, "global": 100}, "feature_set": best_set, "final_model": final_name,
                  "versions": {"sklearn": sklearn.__version__, "lightgbm": lgb.__version__}}
json.dump(feature_config, open("export/models/feature_config.json", "w"), indent=1)
json.dump(history_snapshot(data), open("export/models/history_snapshot.json", "w"))

display_cols = ["Country", "Shipment Mode", "Product Group"]
upcoming = test[["ID", "Vendor", "Scheduled Delivery Date"] + list(dict.fromkeys(display_cols + FEATURES)) + ["target_late"]].copy()
upcoming = upcoming.rename(columns={"ID": "shipment_id", "Vendor": "vendor", "Scheduled Delivery Date": "scheduled_date",
                                    "target_late": "actual_late"})
upcoming["scheduled_date"] = upcoming["scheduled_date"].dt.strftime("%Y-%m-%d")
upcoming.to_csv("export/data/upcoming_shipments.csv", index=False)

import shutil
shutil.copy("figures/06_model_comparison.png", "export/figures_06_model_comparison.png")
shutil.copy("figures/07_calibration.png", "export/figures_07_calibration.png")
shutil.copy("figures/08_shap_summary.png", "export/figures_08_shap_summary.png")
shutil.make_archive("pulsechain_model_export", "zip", "export")
files.download("pulsechain_model_export.zip")
print("Install these versions locally if the app cannot load the model:")
print("pip install scikit-learn==%s lightgbm==%s" % (sklearn.__version__, lgb.__version__))

## 10. Summary for the defense

In [ ]:
best = results.loc[final_name]
rule = results.loc["Rule: vendor recent late rate"]
print("Final model:", final_name, "| feature set:", best_set)
print("Test ROC-AUC: final model %.3f vs vendor-recent-rate rule %.3f" % (best["ROC-AUC"], rule["ROC-AUC"]))
print("Reviewing the top 20%% of shipments catches %.0f%% of late ones (%.1fx better than random); the rule catches %.0f%%" % (best["Top-20% recall"] * 100, best["Lift@20%"], rule["Top-20% recall"] * 100))
print("At the 80%%-recall threshold: precision %.2f, recall %.2f, flags %.0f%% of shipments" % (best["Precision"], best["Recall"], best["Flagged share"] * 100))